# Counting Stability Chambers of Moduli Spaces

This notebook shows how this partitioning algorithm is useful in **multiple areas of interest**.
In particular, it has been used in **state-of-the-art research on moduli spaces of parabolic vector bundles**,
where it allowed us to **count the number of stability chambers** of these moduli spaces with parameters $n$ (number of stability points) and $r$ (rank of vector bundles).

For background see:

> [_A computational analysis of isomorphism classes of moduli spaces of parabolic vector bundles_](https://repositorio.comillas.edu/xmlui/handle/11531/89959)


## Step 1. Imports and Setup


In [6]:
from time import perf_counter

from polypart import build_partition_tree
from polypart.generators import get_product_of_simplices, get_moduli_arrangement
from polypart import save_tree, load_tree

## Step 2. Build the Polytope

In a moduli space of parabolic vector bundles, the ambient space is a product of simplices of dimension $n(r-1)$.
We can build this polytope using the predefined function `polypart.polytopes.get_simplex`.

We will study the case of $n=1$ and $r=7$, which gives a simplex of dimension $6$ with $7$ facets.


In [7]:
n, r = 1, 7
simplex = get_product_of_simplices(n, r - 1)
simplex.extreme()
simplex

Polytope(dim=6, n_ineq=7, n_vertices=7)

## Step 3. Generate Hyperplane Arrangement

In the moduli space classification problem, there are some hyperplanes called "stability walls" that partition the parameter space (starting polytope) into chambers, where each chamber represents a different moduli space.

We will generate these hyperplanes using the predefined function `polypart.arrangements.get_moduli_arrangement`, which implements the combinatorial construction of these hyperplanes for the given parameters $n$, $r$, and $d$ (degree of the vector bundles, which we set to $0$ for simplicity).


In [8]:
hyperplanes = get_moduli_arrangement(n, r, d=0)
print(f"Generated arrangement with {len(hyperplanes)} hyperplanes.")

Generated arrangement with 65 hyperplanes.


## Step 4. Build the Partition Tree

We now apply the partitioning algorithm to the polytope and the set of hyperplanes using to different strategies: 'random' and 'v-entropy'. We compare the runtime and the maximum and average depth of the resulting trees.


- 'Random' strategy: selects hyperplanes randomly at each node.


In [13]:
from polypart.experiments.stats import print_ppart_stats

start = perf_counter()
tree, n_chambers = build_partition_tree(
    hyperplanes,
    simplex,
    strategy="random",
    remove_redundancies=True,
    record_stats=True,
)
elapsed = perf_counter() - start
print(f"Found {n_chambers} chambers in {elapsed:.2f} s")

print_ppart_stats(tree)

Found 1296 chambers in 1.16 s
{
    "total_nodes": 2591,
    "leaf_count": 1296,
    "avg_depth": 15.0255,
    "max_depth": 24,
    "has_detailed_stats": true,
    "avg_candidates": 2.5384,
    "avg_inequalities": 7.9757,
    "avg_vertices": 8.8472
}


- 'V-entropy' strategy: maximizes volume entropy (approximated with number of vertices in each sub-polytope).


In [5]:
start = perf_counter()
tree, n_chambers = build_partition_tree(
    hyperplanes, simplex, strategy="v-entropy", remove_redundancies=True
)
elapsed = perf_counter() - start
print(f"Found {n_chambers} chambers in {elapsed:.2f} s")

# print_ppart_stats(tree)

KeyboardInterrupt: 

## Step 6. Saving the Tree (Optional)

The decision tree is very useful in research for identifying **isomorphisms** between stability chambers, since it allows for efficient point location queries, and it also allows for a detailed analysis of the structure of the partition.


In [6]:
save_tree(tree, f"./data/moduli_tree_n{n}_r{r}.json")


In [7]:
tree = load_tree(f"./data/moduli_tree_n{n}_r{r}.json")

# Print full tree to verify correct loading
# root = tree.root
# queue = [root]
# while queue:
#     node = queue.pop(0)
#     print(node)
#     print("attrs:", node.__dict__.keys())
#     queue.extend(node.children)
#     if node.data and "centroid" in node.data:
#         print("centroid:", node.data["centroid"])

In [8]:
from functools import partial

import numpy as np

from polypart.core.typing import FractionVector, as_fraction_vector
from polypart.algorithms.graph import (
    SymmetryGroup,
    PartitionGraph,
    StorageLevel,
)
from polypart.apps.moduli import (
    basic_transformation,
    generate_d_invariant_transformations,
)


def _apply_wrapper(
    vector: FractionVector,
    sigma: tuple[int, ...],
    s: int,
    H: tuple[int, ...],
    n: int,
    r: int,
    d: int,
) -> FractionVector:
    """Reshape flat seed vector, apply transformation, flatten back."""
    zeros = as_fraction_vector(np.zeros(n)).reshape(n, 1)
    alphas = np.hstack((zeros, vector.reshape(n, r - 1))).reshape(1, n, r)
    d_arr = np.array([d])

    new_alphas, _ = basic_transformation(alphas, d_arr, sigma, s, H)

    return new_alphas[0, :, 1:].flatten()


def get_moduli_symmetries(n: int, r: int, d: int) -> SymmetryGroup:
    """Create SymmetryGroup from moduli space transformations."""
    transforms = {}
    generator = generate_d_invariant_transformations(n, r, d)

    for i, (sigma, s, H) in enumerate(generator):
        name = f"T_{i}_sig{sigma}_s{s}_H{H}"
        func = partial(_apply_wrapper, sigma=sigma, s=s, H=H, n=n, r=r, d=d)
        transforms[name] = func

    return SymmetryGroup(transforms)


symmetries = get_moduli_symmetries(n, r, 0)
print(f"Generated symmetry group with {len(symmetries)} transformations.")

# Create partition graph and reduce to equivalence classes
graph = PartitionGraph(tree, symmetries)
n_classes = graph.reduce(storage=StorageLevel.FULL, verbose=True)

# for eq_class in graph.classes:
#     print(f"Class {eq_class.id}: {eq_class.size} cells")
#     print(f"  Seed: {eq_class.seed_node.data['centroid']}")
#     print(f"  Stabilizers: {eq_class.stabilizers}")
#     print(f"  Mappings: {eq_class.mappings}")
print(f"Reduced to {n_classes} equivalence classes under symmetries.")

Generated symmetry group with 16 transformations.
Reducing 640 cells with 16 symmetries...


Computing classes: 100%|██████████| 640/640 [00:00<00:00, 5871.56it/s]

Found 44 equivalence classes in 0.11s
Reduced to 44 equivalence classes under symmetries.
